In [10]:
import pandas as pd
import re
import html
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")

# Keep negations because they can change the emotion expressed.
stop_words = set(stopwords.words("english")) - {"no", "nor", "not", "never", "cannot"}
tokenizer = TweetTokenizer()
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [23]:
from datasets import load_dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "emotion")

train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

In [12]:
%pip install emoji contractions

Note: you may need to restart the kernel to use updated packages.


In [19]:
def clean_text(text):
    """
    Basic tweet preprocessing.
    """
    
    import contractions
    
    # Expand forms such as "don't" to "do not".
    text = contractions.fix(text)
    
    # Convert to lowercase
    text = html.unescape(text).lower()
    
    # Replace URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Replace user mentions
    text = re.sub(r"@\w+", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    import emoji

    # "I love this ❤️" -> "I love this red_heart"
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # Keep hashtag words, but remove the # symbol
    text = re.sub(r"#(\w+)", r"\1", text)
    
    # Remove punctuation and symbols while keeping words and numbers.
    text = re.sub(r"[^\w\s]|_", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    tokens = tokenizer.tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    
    pos_map = {
        "J": wordnet.ADJ,
        "V": wordnet.VERB,
        "N": wordnet.NOUN,
        "R": wordnet.ADV,
    }
    tagged_tokens = nltk.pos_tag(tokens)
    tokens = [
        lemmatizer.lemmatize(word, pos_map.get(tag[0], wordnet.NOUN))
        for word, tag in tagged_tokens
    ]
    
    return " ".join(tokens)

In [27]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
val_df["clean_text"] = val_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [21]:
comparison = train_df[["text", "clean_text"]].head(10)

for _, row in comparison.iterrows():
    print("Original:", row["text"])
    print("Cleaned: ", row["clean_text"])
    print("-" * 80)

Original: “Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry
Cleaned:  worry payment problem may never joyce meyer motivation leadership worry
--------------------------------------------------------------------------------
Original: My roommate: it's okay that we can't spell because we have autocorrect. #terrible #firstworldprobs
Cleaned:  roommate okay cannot spell autocorrect terrible firstworldprobs
--------------------------------------------------------------------------------
Original: No but that's so cute. Atsu was probably shy about photos before but cherry helped her out uwu
Cleaned:  no cute atsu probably shy photo cherry help uwu
--------------------------------------------------------------------------------
Original: Rooneys fucking untouchable isn't he? Been fucking dreadful again, depay has looked decent(ish)tonight
Cleaned:  rooneys fuck untouchable not fuck dreadful depay look decent ish tonight
------------------

In [28]:
import os

raw_dir = "../data/raw"
preprocessed_dir = "../data/preprocessed"
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(preprocessed_dir, exist_ok=True)

for split_name, df in {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}.items():
    df[["text", "label"]].to_csv(
        f"{raw_dir}/{split_name}.csv",
        index=False,
        encoding="utf-8",
    )
    df[["clean_text", "label"]].to_csv(
        f"{preprocessed_dir}/{split_name}.csv",
        index=False,
        encoding="utf-8",
    )